In [1]:
# Py Packages Install
!pip install scikit-allel numpy pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 10.6 MB/s eta 0:00:0000:0100:01


In [2]:
# Required Modules
import allel
import numpy as np
import pandas as pd
import random
from pathlib import Path

### 1. SNP Sequences

SNPs (Single Nucleotide Polymorphisms) are the most common type of genetic variation among people.
They represent a difference in a single DNA building block (nucleotide) — A, T, C, or G — at a specific position in the genome.
- filtering some important SNPs for downstream analysis.

Useful SNPs for personalized nutrition

rs429358 & rs7412 (APOE pair). APOE genotype (E2/E3/E4) strongly influences lipid metabolism and response to dietary fat. Highly relevant to dietary fat recommendations and cardiometabolic risk modeling.

rs1801133 (MTHFR C677T). Affects folate metabolism, homocysteine — relevant to micronutrient recommendations (folate, B12) and some cardiovascular risk modelling. Clinical utility is debated for disease prediction, but for dietary micronutrient personalization it’s useful.

rs9939609 (FTO). Classic obesity/metabolism locus — useful for modeling weight-related response to diet, albeit with small effect size. Good for population-level risk features in an ML model.

In [3]:
SNP_PANEL_DEFAULT = [
    {"rsid": "rs429358", "gene": "APOE", "uniprot": "P02649"},
    {"rsid": "rs7412", "gene": "APOE", "uniprot": "P02649"},
    {"rsid": "rs1801133", "gene": "MTHFR", "uniprot": "P42898"},
    {"rsid": "rs9939609", "gene": "FTO",  "uniprot": "Q86ZP4"},
]

# reference lookup panel -> connecting genetic variants → genes → proteins

# SNP ID (rsid) → dbSNP database
# Gene → functional genomic context
# UniProt ID → protein-level annotation according to gene

**Downloading VCFs from 1000 Genomeproject for SNP Seq**
- https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/
- The data in this collection represents the final work of the 1000 Genomes Project, as completed in phase three of the project on GRCh37
- Based on above taken snp default genes
- rs429358 and rs7412 → APOE → chromosome 19.
- rs1801133 → MTHFR → chromosome 1.
- rs9939609 → FTO → chromosome 16.

In [4]:
import glob
vcf_paths = glob.glob("/kaggle/input/genomevcf/*.vcf")
vcf_paths

['/kaggle/input/genomevcf/ALL.chr16.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf',
 '/kaggle/input/genomevcf/ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf',
 '/kaggle/input/genomevcf/ALL.chr19.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf']

In [5]:
# Helper Functions

def read_vcf_biallelic_snps(vcf_path, alt_number=1):
    rec = allel.read_vcf(str(vcf_path),
                         fields=['samples', 'variants/CHROM', 'variants/POS',
                                 'variants/REF', 'variants/ALT', 'calldata/GT'],
                         alt_number=alt_number)
    if rec is None or 'calldata/GT' not in rec:
        return None, [], pd.DataFrame()

    samples = list(rec['samples'])
    chrom = np.asarray(rec['variants/CHROM'])
    pos = np.asarray(rec['variants/POS'])
    ref = np.asarray(rec['variants/REF'])
    alt = np.asarray(rec['variants/ALT'])

    def get_first_alt(a):
        try:
            first = a[0]
        except Exception:
            first = a
        if isinstance(first, bytes):
            return first.decode()
        return first

    alt0 = np.array([get_first_alt(a) for a in alt], dtype=object)

    is_snp = np.array([isinstance(r, (str, bytes)) and len(r) == 1 and
                       isinstance(a, str) and a is not None and len(a) == 1
                       for r, a in zip(ref, alt0)], dtype=bool)

    gt = rec['calldata/GT']
    minlen = min(gt.shape[0], len(is_snp))
    gt, is_snp, chrom, pos, ref, alt0 = gt[:minlen], is_snp[:minlen], chrom[:minlen], pos[:minlen], ref[:minlen], alt0[:minlen]

    gt_snp = gt[is_snp]
    variants_df = pd.DataFrame({
        'CHROM': chrom[is_snp].astype(str),
        'POS': pos[is_snp].astype(int),
        'REF': ref[is_snp].astype(str),
        'ALT': alt0[is_snp].astype(str)
    })

    return gt_snp, samples, variants_df


def align_and_concatenate(vcf_paths, alt_number=1):
    per_file = []
    sample_sets = []
    for p in vcf_paths:
        gt, samples, vars_df = read_vcf_biallelic_snps(p, alt_number=alt_number)
        per_file.append((gt, samples, vars_df))
        sample_sets.append(samples)

    samples_union = []
    for s_list in sample_sets:
        for s in s_list:
            if s not in samples_union:
                samples_union.append(s)

    n_union = len(samples_union)
    ploidy = 2
    gt_arrays = []
    var_dfs = []

    for gt, samples, vars_df in per_file:
        if gt.size == 0:
            continue
        n_variants = gt.shape[0]
        gt_full = np.full((n_variants, n_union, ploidy), -1, dtype=int)
        sample_to_idx = {s: i for i, s in enumerate(samples)}
        for j, s in enumerate(samples_union):
            if s in sample_to_idx:
                gt_full[:, j, :] = gt[:, sample_to_idx[s], :]
        gt_arrays.append(gt_full)
        var_dfs.append(vars_df)

    if not gt_arrays:
        return np.empty((0, n_union, ploidy), dtype=int), samples_union, pd.DataFrame()

    gt_all = np.concatenate(gt_arrays, axis=0)
    variants_concat = pd.concat(var_dfs, ignore_index=True)
    return gt_all, samples_union, variants_concat


def sort_variants_and_reorder_gt(gt_all, variants_df):
    if variants_df.empty:
        return gt_all, variants_df
    variants_df = variants_df.copy()
    variants_df['_idx'] = np.arange(len(variants_df))

    def chrom_key(c):
        try:
            return (0, int(c))
        except Exception:
            return (1, c)

    chrom_keys = [chrom_key(c) for c in variants_df['CHROM']]
    sort_df = pd.DataFrame({
        'chrom_key_0': [k[0] for k in chrom_keys],
        'chrom_key_1': [k[1] for k in chrom_keys],
        'POS': variants_df['POS'],
        'idx': variants_df['_idx']
    })
    sorted_idx = sort_df.sort_values(['chrom_key_0', 'chrom_key_1', 'POS'])['idx'].values
    gt_sorted = gt_all[sorted_idx, ...]
    variants_sorted = variants_df.loc[sorted_idx].reset_index(drop=True).drop(columns=['_idx'])
    return gt_sorted, variants_sorted


def genotype_to_n_alt(gt_array):
    g = allel.GenotypeArray(gt_array)
    return g.to_n_alt(fill=-1)


def generate_sequences(n_alt, M=4983, L=100, contiguous=True, rng_seed=42, missing_replace=0):
    random.seed(rng_seed)
    np.random.seed(rng_seed)
    n_variants, n_samples = n_alt.shape
    sequences = np.empty((M, L), dtype=int)
    for i in range(M):
        s_idx = random.randrange(n_samples)
        if contiguous and n_variants >= L:
            start = random.randrange(0, n_variants - L + 1)
            seq = n_alt[start:start+L, s_idx]
        else:
            idxs = np.random.choice(n_variants, size=L, replace=(n_variants < L))
            seq = n_alt[idxs, s_idx]
        seq = np.where(seq == -1, missing_replace, seq)
        sequences[i] = seq.astype(int)
    return sequences


def main(vcf_files, out_csv='snp_sequences.csv', M=4983, L=100, rng_seed=1234):
    print("Reading and aligning VCFs:", vcf_files)
    gt_all, samples_union, variants_concat = align_and_concatenate(vcf_files)
    print(f"Loaded {gt_all.shape[0]} variants across {len(samples_union)} samples.")
    gt_sorted, variants_sorted = sort_variants_and_reorder_gt(gt_all, variants_concat)
    n_alt = genotype_to_n_alt(gt_sorted)
    sequences = generate_sequences(n_alt, M=M, L=L, contiguous=True, rng_seed=rng_seed)

    print("Writing sequences to CSV...")
    seq_df = pd.DataFrame(sequences, columns=[f"Snp_{i+1}" for i in range(L)])
    seq_df.index = [f"Seq_{i+1:05d}" for i in range(M)]
    seq_df.to_csv(out_csv, index=True)
    print(f"✅ Saved {M} sequences (length={L}) to {out_csv}")
    print("Preview:")
    display(seq_df.head())
    return seq_df

In [6]:
# Run of above main function to generate snpseq
snpseq = main(vcf_paths)

Reading and aligning VCFs: ['/kaggle/input/genomevcf/ALL.chr16.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf', '/kaggle/input/genomevcf/ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf', '/kaggle/input/genomevcf/ALL.chr19.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf']
Loaded 8 variants across 2504 samples.
Writing sequences to CSV...
✅ Saved 4983 sequences (length=100) to snp_sequences.csv
Preview:


,Snp_1,Snp_2,Snp_3,Snp_4,Snp_5,Snp_6,Snp_7,Snp_8,Snp_9,Snp_10,...,Snp_91,Snp_92,Snp_93,Snp_94,Snp_95,Snp_96,Snp_97,Snp_98,Snp_99,Snp_100
Seq_00001,0,0,0,0,0,0,1,0,0,0,...,1,0,0,0,1,0,0,1,0,0
Seq_00002,0,0,0,0,0,0,0,0,0,0,...,2,0,0,1,1,0,0,0,0,1
Seq_00003,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
Seq_00004,0,0,1,1,0,1,1,0,0,0,...,0,0,0,0,0,1,1,1,1,1
Seq_00005,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0


* 100 Snps into one List

In [7]:
# Path to your existing SNP sequences CSV
input_csv = "/kaggle/working/snp_sequences.csv"
output_csv = "final_snp_sequences.csv"

# Read the CSV
df = pd.read_csv(input_csv, index_col=0)

# Identify only SNP columns (those starting with 'snp_')
snp_cols = [c for c in df.columns if c.startswith("Snp_")]

# Option 1: as list strings
df['Snp_Sequence'] = df[snp_cols].apply(lambda row: row.tolist(), axis=1)

# Keep only what you need
result_df = df[['Snp_Sequence']]

# Save to new CSV
result_df.to_csv(output_csv, index_label='Seq_Id')

print(f"✅ Saved flattened SNP sequences to {output_csv}")
display(result_df.head())

✅ Saved flattened SNP sequences to final_snp_sequences.csv


,Snp_Sequence
Seq_00001,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, ..."
Seq_00002,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, ..."
Seq_00003,"[0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, ..."
Seq_00004,"[0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, ..."
Seq_00005,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, ..."


### 2. Protein Sequence Foldings

In [8]:
import pandas as pd
import math
import ast

# === Step 1: Load SNP sequences ===
input_path = "/kaggle/working/final_snp_sequences.csv"   # your SNP input file
snp_df = pd.read_csv(input_path)

# Parse the SNP list column from string to list
snp_df["Snp_Sequence"] = snp_df["Snp_Sequence"].apply(ast.literal_eval)

# === Step 2: Define deterministic mapping ===
# SNP values mapped to protein-like base embeddings
base_map = {0: 0.68, 1: 0.75, 2: 0.82}

def deterministic_protein_embeddings(snp_seq):
    """
    Returns two deterministic embeddings per SNP element:
      1. Protein_Sequence (residue-like embedding scalar per position)
      2. Protein_Sequence_Folds (fold-stability scalar per position)
    """
    seq_emb = []
    fold_emb = []
    for i, v in enumerate(snp_seq):
        base = base_map.get(v, 0.70)
        # small deterministic position-dependent variation
        variation_seq = 0.02 * math.sin((i + 1) * 0.5 + v * 0.7)
        variation_fold = -0.01 * math.cos((i + 1) * 0.3 + v * 0.5)
        seq_emb.append(round(base + variation_seq, 2))
        fold_emb.append(round(base + variation_fold, 2))
    return seq_emb, fold_emb

# === Step 3: Apply transformation to all SNP rows ===
protein_seqs = []
protein_folds = []
for seq in snp_df["Snp_Sequence"]:
    seq_emb, fold_emb = deterministic_protein_embeddings(seq)
    protein_seqs.append(seq_emb)
    protein_folds.append(fold_emb)

# === Step 4: Add to dataframe ===
# snp_df["Protein_Sequence"] = protein_seqs
snp_df["Protein_Sequence_Folds"] = protein_folds

# === Step 5: Save output ===
output_path = "Snp_Seqs+Protein_Seq_Folds.csv"
snp_df.to_csv(output_path, index=False)

print(f"✅ Deterministic protein folding dataset saved to: {output_path}")
display(snp_df.head(3))

✅ Deterministic protein folding dataset saved to: Snp_Seqs+Protein_Seq_Folds.csv


,Seq_Id,Snp_Sequence,Protein_Sequence_Folds
0,Seq_00001,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0.67, 0.67, 0.67, 0.68, 0.68, 0.68, 0.76, 0.6..."
1,Seq_00002,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, ...","[0.67, 0.67, 0.67, 0.68, 0.68, 0.68, 0.69, 0.6..."
2,Seq_00003,"[0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, ...","[0.67, 0.67, 0.67, 0.68, 0.75, 0.76, 0.69, 0.6..."
